# Phase 5 — node dataset audit and annotation preparation

This notebook performs a **read-only** inventory of a locally synced acquisition folder. It does not annotate, train, alter raw images, run SAM2, or create pseudo-labels.

## Annotation scope

Annotate only developed leaf-origin junctions on the target pea plant's single main vertical stem. Assign `Node1 … NodeN` from base to apex. An undeveloped apex is not a node. Use `visible`, `partially_occluded`, or `uncertain`; never place a coordinate for an invisible node. Across views, reuse an index only when the same biological node is reliable. See `docs/node_annotation_protocol_v1.md`.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Run this notebook from the thesis project or its notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.node_dataset_audit import (build_manifest, configured_data_root, discovery_diagnostics, find_unexpected_extensions,
                                    save_contact_sheets, summarise, write_report)

DATA_ROOT = os.environ.get('NODE_DATA_ROOT')  # Set only to a locally synced, read-only Drive folder.
OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'node_dataset_audit'
DATA_ROOT

## Configure the source location

No Google Drive folder was assumed or copied. Set `NODE_DATA_ROOT` to the local sync location containing the date/plant folders, then run the next cells. Example in a terminal: `NODE_DATA_ROOT='/path/to/synced/acquisitions' python scripts/run_node_dataset_audit.py --data-root "$NODE_DATA_ROOT"`.

In [ ]:
if not DATA_ROOT:
    print('DATA_ROOT is not configured. No audit outputs have been generated.')
    print('Set NODE_DATA_ROOT, then rerun from this cell. Raw images will remain read-only.')
else:
    data_root = configured_data_root(DATA_ROOT)
    diagnostics = discovery_diagnostics(data_root)
    print(f'Read-only DATA_ROOT: {data_root}')
    print(f"DATA_ROOT exists: {diagnostics['data_root_exists']}")
    print(f"DATA_ROOT is directory: {diagnostics['data_root_is_directory']}")
    print(f"raw recursive supported-image count before filtering: {diagnostics['raw_recursive_supported_image_count']}")
    print(f"supported-image count after filtering: {diagnostics['supported_image_count_after_filtering']}")

In [ ]:
if DATA_ROOT:
    manifest = build_manifest(data_root)
    summary, anomalies = summarise(manifest)
    unsupported = find_unexpected_extensions(data_root)
    source_issues = []
    for record in manifest.loc[~manifest.readable].itertuples():
        source_issues.append({'acquisition_date': record.acquisition_date, 'plant_id': record.plant_id, 'audit_status': 'unreadable_image', 'source_path': record.source_path})
    for record in manifest.loc[manifest.duplicate_hash].itertuples():
        source_issues.append({'acquisition_date': record.acquisition_date, 'plant_id': record.plant_id, 'audit_status': 'duplicate_file_hash', 'source_path': record.source_path, 'filename': record.filename})
    duplicate_names = manifest.duplicated(['acquisition_date', 'plant_id', 'filename'], keep=False)
    for record in manifest.loc[duplicate_names].itertuples():
        source_issues.append({'acquisition_date': record.acquisition_date, 'plant_id': record.plant_id, 'audit_status': 'duplicate_filename_within_plant_date', 'source_path': record.source_path, 'filename': record.filename})
    for record in manifest.loc[manifest.plant_id.isna() | manifest.acquisition_date.isna()].itertuples():
        source_issues.append({'acquisition_date': record.acquisition_date, 'plant_id': record.plant_id, 'audit_status': 'missing_or_unexpected_path_metadata', 'source_path': record.source_path})
    for record in unsupported.itertuples():
        source_issues.append({'audit_status': 'unexpected_extension', 'source_path': record.source_path, 'filename': record.filename, 'extension': record.extension})
    if source_issues:
        anomalies = pd.concat([anomalies, pd.DataFrame(source_issues)], ignore_index=True, sort=False)
    tables = OUTPUT_ROOT / 'tables'
    tables.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(tables / 'node_dataset_manifest.csv', index=False)
    summary.to_csv(tables / 'dataset_summary_by_date_plant.csv', index=False)
    anomalies.to_csv(tables / 'acquisition_anomalies.csv', index=False)
    candidates = manifest.loc[manifest.plant_id.isin(['P07', 'P12']), ['acquisition_date', 'plant_id', 'view_slot', 'filename', 'source_path', 'width', 'height', 'mean_brightness', 'mean_saturation', 'sharpness_laplacian_variance']].copy()
    candidates['selected_for_cvat'] = ''
    candidates['selection_reason'] = ''
    candidates.to_csv(tables / 'pilot_annotation_candidates.csv', index=False)
    save_contact_sheets(manifest, OUTPUT_ROOT / 'contact_sheets')
    write_report(data_root, manifest, summary, anomalies, OUTPUT_ROOT / 'dataset_audit_report.md')
    display(summary)
    display(anomalies)
    display(unsupported)

## Interpretation boundary

The expected inventory (483 images across the four sessions) is an audit target, not an ML configuration. September 29 plant-specific counts are retained for manual review. P06 is deliberately marked `dead`; its one image per session is expected. `view_slot` is chronological neutral metadata and `view_label` remains `unknown` unless trusted acquisition metadata becomes available.

## Future planning only

For a later manually reviewed node-keypoint study, the intended planning estimate is 19 living plants × 4 dates × ~2 selected views = ~152 images, with a plant-level split near 13/3/3. A future ResNet heatmap model is only a planning item; no model or training is implemented here.